# MACCS Encoder

In [3]:
# ==============================
# Unsupervised FCN Autoencoder

# ==============================

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import random

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from torch.utils.data import Dataset, DataLoader


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


def GetPubChemFPs(mol):
    from rdkit.Chem import rdMolDescriptors
    return rdMolDescriptors.GetHashedAtomPairFingerprintAsBitVect(
        mol,
        nBits=881
    )

def extract_fingerprints(smiles: str) -> torch.Tensor:

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError(f"SMILES ไม่ถูกต้อง: {smiles}")

    # ---------------- MACCS ----------------
    fp_maccs_rd = AllChem.GetMACCSKeysFingerprint(mol)

    fp_maccs = np.zeros(
        (fp_maccs_rd.GetNumBits(),),
        dtype=int
    )

    DataStructs.ConvertToNumpyArray(
        fp_maccs_rd,
        fp_maccs
    )

    fingerprint = fp_maccs

    return torch.tensor(
        fingerprint,
        dtype=torch.float32
    )


class SMILESDataset(Dataset):

    def __init__(self, smiles_list):
        self.smiles_list = smiles_list

    def __len__(self):
        return len(self.smiles_list)

    def __getitem__(self, idx):

        fp = extract_fingerprints(
            self.smiles_list[idx]
        )

        return fp


class FCNAutoencoder(nn.Module):

    def __init__(self, input_dim, latent_dim=64):

        super().__init__()

        # -------- Encoder --------
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, latent_dim)
        )

        # -------- Decoder --------
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 256),
            nn.ReLU(),

            nn.Linear(256, input_dim)
        )

    def forward(self, x):

        latent = self.encoder(x)

        recon = self.decoder(latent)

        return latent, recon


def train_fcn_autoencoder(
    smiles_list,
    latent_dim=128,
    epochs=50,
    batch_size=32,
    lr=1e-3
):

    dataset = SMILESDataset(smiles_list)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True
    )

    input_dim = extract_fingerprints(
        smiles_list[0]
    ).shape[0]

    model = FCNAutoencoder(
        input_dim=input_dim,
        latent_dim=latent_dim
    )

    optimizer = optim.Adam(
        model.parameters(),
        lr=lr
    )

    criterion = nn.MSELoss()

    model.train()

    for epoch in range(epochs):

        epoch_loss = 0

        for fp in loader:

            optimizer.zero_grad()

            latent, recon = model(fp)

            loss = criterion(recon, fp)

            loss.backward()

            optimizer.step()

            epoch_loss += loss.item()

        if (epoch + 1) % 10 == 0:

            print(
                f"Epoch {epoch+1}/{epochs}, "
                f"Loss: {epoch_loss/len(loader):.4f}"
            )

    return model


def generate_embedding(
    df,
    model,
    out_csv
):

    model.eval()

    embeddings = []

    with torch.no_grad():

        for smi in df["Smiles"]:

            try:

                fp = extract_fingerprints(
                    smi
                ).unsqueeze(0)

                latent, _ = model(fp)

                embeddings.append(
                    latent.squeeze(0)
                    .cpu()
                    .numpy()
                )

            except:

                embeddings.append(
                    np.zeros(
                        model.encoder[-1].out_features
                    )
                )

    emb_df = pd.DataFrame(embeddings)

    emb_df.to_csv(
        out_csv,
        index=False
    )


def run_pipeline(
    train_path,
    test_path,
    out_train_csv="Embedding_train.csv",
    out_test_csv="Embedding_test.csv",
    latent_dim=64,
    epochs=50
):

    # -------- Load train --------
    df_train = pd.read_csv(train_path)

    train_smiles = (
        df_train["Smiles"]
        .astype(str)
        .tolist()
    )

    # -------- Train autoencoder --------
    model = train_fcn_autoencoder(
        train_smiles,
        latent_dim=latent_dim,
        epochs=epochs
    )

    # -------- Generate train embeddings --------
    generate_embedding(
        df_train,
        model,
        out_train_csv
    )

    # -------- Generate test embeddings --------
    df_test = pd.read_csv(test_path)

    generate_embedding(
        df_test,
        model,
        out_test_csv
    )


run_pipeline(
    train_path="NAI2_Train.csv",

    test_path="NAI2_Test.csv",

    out_train_csv="maccs-Embed_train.csv",

    out_test_csv="maccs-Embed_test.csv",

    latent_dim=64,

    epochs=200
)

Epoch 10/200, Loss: 0.0445
Epoch 20/200, Loss: 0.0259
Epoch 30/200, Loss: 0.0183
Epoch 40/200, Loss: 0.0145
Epoch 50/200, Loss: 0.0122
Epoch 60/200, Loss: 0.0103
Epoch 70/200, Loss: 0.0092
Epoch 80/200, Loss: 0.0079
Epoch 90/200, Loss: 0.0070
Epoch 100/200, Loss: 0.0065
Epoch 110/200, Loss: 0.0059
Epoch 120/200, Loss: 0.0053
Epoch 130/200, Loss: 0.0048
Epoch 140/200, Loss: 0.0046
Epoch 150/200, Loss: 0.0043
Epoch 160/200, Loss: 0.0041
Epoch 170/200, Loss: 0.0038
Epoch 180/200, Loss: 0.0035
Epoch 190/200, Loss: 0.0034
Epoch 200/200, Loss: 0.0033


In [2]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 50.4 MB/s eta 0:00:00:00:0100:01
